<a href="https://colab.research.google.com/github/mf2056/F20AA/blob/main/DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas textblob vaderSentiment scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.6 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving amazon_culture_data.csv to amazon_culture_data.csv


In [4]:
from google.colab import files
uploaded = files.upload()

Saving youtube_amazon_employee.csv to youtube_amazon_employee.csv


In [61]:
import pandas as pd

df1 = pd.read_csv('amazon_culture_data.csv')
df2 = pd.read_csv('youtube_amazon_employee.csv')
df1 = df1[["text"]]
df2 = df2[["text"]]
df = pd.concat([df1, df2], ignore_index=True)

df.to_csv("dataset.csv", index=False)
df.head()

,text
0,I'm suspended for having my Medical Marijuana ...
1,Once again you enlighten US!\nKeep up the grea...
2,Don't ask me to work on my day off and you bet...
3,I'm a proud Amazon Prime member. I'm also atte...
4,Amazon employees are over worked with mandator...


In [62]:
#Basic Cleaning

comment_col = "text"
df = df.dropna(subset=[comment_col])  # Drop rows with missing comments
df[comment_col] = df[comment_col].astype(str).str.strip()

# Remove empty strings
df = df[df[comment_col] != ""]

# Remove duplicates
df = df.drop_duplicates(subset=[comment_col])

# Remove very short comments (less than 3 words)
word_count = df["text"].astype(str).str.split().str.len()
df = df[word_count >= 3]

df.shape

(5547, 1)

In [63]:
# Remove spam comments

spam_keywords = [
    "subscribe", "giveaway", "click", "telegram", "whatsapp",
    "contact me", "dm me", "crypto", "bitcoin", "forex", "trading",
    "please like", "anyone watching in", "bit.ly", "goo.gl",
    "link in bio", "visit my site"
]

def remove_spam(text):
    text_lower = text.lower()
    return not any(word in text_lower for word in spam_keywords)

df = df[df[comment_col].apply(remove_spam)]
df.shape

(5524, 1)

In [64]:
# Filtering relevant comments

workplace_keywords = [
    "employee", "worker", "staff", "associate", "warehouse",
    "leadership", "executive", "perks", "insurance",
    "fulfillment", "manager", "management", "boss",
    "hr", "supervisor", "shift", "overtime", "break", "pay",
    "salary", "wage", "benefits", "culture",
    "workload", "stress", "burnout", "treatment",
    "working", "job", "career", "fired", "hired", "workplace",
    "company", "office", "corporate", "hiring", "recruitment",
    "interview", "promotion", "resignation", "quit", "layoff",
]

def is_relevant(text):
    text_lower = text.lower()
    return any(word in text_lower for word in workplace_keywords)

df = df[df[comment_col].apply(is_relevant)]
df.shape

(3385, 1)

In [65]:
# Textblob labeling

from textblob import TextBlob

def textblob_polarity(text):
    return TextBlob(text).sentiment.polarity

tb_polarity = df["text"].apply(textblob_polarity)

def tb_to_label(p):
    if p > 0.05:
        return "positive"
    elif p < -0.05:
        return "negative"
    else:
        return "neutral"

df['tb_label'] = tb_polarity.apply(tb_to_label)
df['tb_label'].value_counts()

,count
tb_label,
positive,1470
neutral,1101
negative,814


In [66]:
# Sentiment Analysis

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    return analyzer.polarity_scores(text)['compound']

vader_scores= df[comment_col].apply(get_vader_scores)

def score_to_label(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

df["vader_label"] = vader_scores.apply(score_to_label)

df["vader_label"].value_counts()

,count
vader_label,
positive,1777
negative,1204
neutral,404


In [67]:
# Checking how much similar
agreement = df[df["tb_label"] == df["vader_label"]]
disagreement = df[df["tb_label"] != df["vader_label"]]
print("Agreement:", agreement.shape[0])
print("Disagreement:",disagreement.shape[0])



Agreement: 1861
Disagreement: 1524


In [68]:
agreement_sample = agreement.sample(300, random_state=6)
disagreement_sample = disagreement.sample(300, random_state=7)

manual_set = pd.concat([agreement_sample, disagreement_sample])
manual_set = manual_set.drop_duplicates(subset=["text"])
manual_set.to_csv("manual_set.csv", index=False)

In [69]:
#from sklearn.model_selection import train_test_split